In [4]:
import json
import pickle
import numpy as np
import re
import os
import matplotlib.pyplot as plt

def _get_list_or_empty(data_dict, key,yaw):
    value = data_dict.get(key, [])
    if isinstance(value, np.ndarray):
        value = value.tolist()
    # 数据处理逻辑
    value = [[-x, -y] for x, y in value]
    yaw = -(1.57-yaw)
    # 构建旋转矩阵
    rotation_matrix = np.array([
        [np.cos(yaw), np.sin(yaw)],
        [-np.sin(yaw), np.cos(yaw)]
    ])
    value = np.array(value)

    # 提取 x 和 y 坐标
    coordinates = value[:, :2]
    # 计算旋转后的坐标
    rotated_coordinates = np.dot(coordinates, rotation_matrix.T)
    value = np.hstack((rotated_coordinates, value[:, 2:]))
    value = value.tolist()

    return value

def read_bin_file(bin_path):
    """
    读取Kitti格式的二进制点云数据.
    
    参数:
        bin_path (str): .bin 文件的路径.
        
    返回:
        points (numpy.ndarray): 点云数据, 形状为 (N, 4) 或 (N, 3), 其中 N 是点的数量.
    """
    scan = np.fromfile(bin_path, dtype=np.float32)
    return scan.reshape((-1, 4))  # Kitti点云数据通常包含4个值: x, y, z, 反射率

def rotate_points(points, angle_degrees):
    """
    以原点为中心旋转点云数据.
    
    参数:
        points (numpy.ndarray): 点云数据, 形状为 (N, 4) 或 (N, 3).
        angle_degrees (float): 旋转角度（度）.
        
    返回:
        rotated_points (numpy.ndarray): 旋转后的点云数据.
    """
    angle_radians = np.radians(angle_degrees)
    rotation_matrix = np.array([
        [np.cos(angle_radians), -np.sin(angle_radians)],
        [np.sin(angle_radians), np.cos(angle_radians)]
    ])
    
    # 仅旋转 x 和 y 坐标
    points[:, :2] = np.dot(points[:, :2], rotation_matrix.T)
    return points

def plot_lidar(points, title="LIDAR Points"):
    """
    绘制LIDAR点云图.
    
    参数:
        points (numpy.ndarray): 点云数据.
        title (str): 图像标题.
    """
    plt.figure(figsize=(10, 8))
    plt.title(title)
    plt.scatter(points[:, 0], points[:, 1], s=0.1, c=points[:, 2], cmap='viridis')
    plt.colorbar(label='Height')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.axis('equal')
    plt.show()

def process_file(input_folder, output_folder, angle_degrees,input_file_name,output_file_name):
    """
    参数:
        input_folder (str): 包含原始点云数据的文件夹路径.
        output_folder (str): 存储处理后点云数据的文件夹路径.
        angle_degrees (float): 旋转角度（度）.
    """
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    filename = input_file_name
    file_path = os.path.join(input_folder, filename)
    lidar_points = read_bin_file(file_path)
    rotated_points = rotate_points(lidar_points, angle_degrees)
    
    # 构建输出文件路径
    output_path = os.path.join(output_folder, output_file_name)
    rotated_points.tofile(output_path)  # 保存旋转后的点云数据到新文件
    
    print(f"Processed and saved {output_file_name} to {output_path}")

# 从 Pickle 文件中读取数据
with open('/home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/data/kitti/data08_491_550.pkl', 'rb') as file:
    data = pickle.load(file)

with open('/home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/data/kitti/data_utm_to_info08.pkl', 'rb') as file:
    data_utm_to_info = pickle.load(file)
# 解包数据
trajectory_hmi_map = data['trajectory_hmi_map']
trajectory_hmi_backward_map = data['trajectory_hmi_backward_map']
utm_pose = data['utm_pose']
trajectory_ins = data['trajectory_ins']
trajectory_ins_past = data['trajectory_ins_past']
utm_to_info = data_utm_to_info['utm_to_info']
# print(f"trajectory_ins {trajectory_ins}")
# print(f"trajectory_ins_past {trajectory_ins_past}")
# print(f"trajectory_hmi_map {trajectory_hmi_map}")
print(f"utm_to_info {len(utm_to_info)}")


for i, pose in enumerate(utm_pose):
    if i < 5 :
        continue
    pose_utm = tuple([pose[0],pose[1]])
    # 处理点云数据
    info = utm_to_info.get(utm_pose[i], None)
    if info is None:
        print(f"utm_pose[i] {utm_pose[i]} not in utm_to_info")
        # 使用正则表达式提取数字部分
    match = re.search(r'\d+', info['filename'])
    if match:
        number_str = match.group()
        number = int(number_str)
    else:
        print("No number found in the filename.")
    input_file_name = f'{number:010}.bin'
    yaw = info['yaw']


    # 替换为实际的经纬度值
    if (pose_utm not in trajectory_ins) and (pose_utm not in trajectory_ins_past) and (pose_utm not in trajectory_hmi_map) and (pose_utm not in trajectory_hmi_backward_map):
        print(f"pose_utm {pose_utm} not in trajectory_ins")
        continue

    A = trajectory_ins.get(pose_utm, []).tolist() if isinstance(trajectory_ins.get(pose_utm, []), np.ndarray) else trajectory_ins.get(pose_utm, [])
    B = trajectory_ins_past.get(pose_utm, []).tolist() if isinstance(trajectory_ins_past.get(pose_utm, []), np.ndarray) else trajectory_ins_past.get(pose_utm, [])
    C = trajectory_hmi_map.get(pose_utm, []).tolist() if isinstance(trajectory_hmi_map.get(pose_utm, []), np.ndarray) else trajectory_hmi_map.get(pose_utm, [])
    D = trajectory_hmi_backward_map.get(pose_utm, []).tolist() if isinstance(trajectory_hmi_backward_map.get(pose_utm, []), np.ndarray) else trajectory_hmi_backward_map.get(pose_utm, [])
    if A is None or B is None or C is None or D is None:
        continue
    if len(A) != 15 or len(B) != 5 or len(C) != 15 or len(D) != 5:
        continue
    output_data = {
        "utm_pose": list(pose_utm),
        "trajectory_ins": _get_list_or_empty(trajectory_ins, pose_utm,yaw),
        "trajectory_ins_past": _get_list_or_empty(trajectory_ins_past, pose_utm,yaw),
        "trajectory_hmi": _get_list_or_empty(trajectory_hmi_map, pose_utm,yaw),
        "trajectory_hmi_past": _get_list_or_empty(trajectory_hmi_backward_map, pose_utm,yaw)
    }



    # 构建文件名，例如 output_0001.json, output_0002.json, ...
    start_idx = 486
    filename = f'{i+1-5+start_idx:06}.json'
    output_file_name = f'{i+1-5+start_idx:06}.bin'

    # if i + 1 -5 == 223:
    #     print(f"pose_utm {pose_utm}")
    #     print(f"input_file_name {input_file_name}")


    output_folder = "/home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/datasets/KITTI_RAW/trajectory_prediction/08" 
    output_path = os.path.join(output_folder, filename)
    # # 将字典转换为 JSON 字符串并写入文件
    with open(output_path, 'w') as json_file:
        print(f"Processed and saved {output_path}")
        
        json.dump(output_data, json_file, indent=4)

    input_folder = "/home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/data/kitti/2011_09_30/2011_09_30_drive_0028_sync/velodyne_points/data"
    
    angle_degrees = 90  # 逆时针旋转的角度
   
    process_file(input_folder, output_folder, angle_degrees,input_file_name,output_file_name)

print("Data has been written to output.json")
# 修改为输出到多个文件中

utm_to_info 1701
Processed and saved /home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/datasets/KITTI_RAW/trajectory_prediction/08/000972.json
Processed and saved 000972.bin to /home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/datasets/KITTI_RAW/trajectory_prediction/08/000972.bin
Processed and saved /home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/datasets/KITTI_RAW/trajectory_prediction/08/000973.json
Processed and saved 000973.bin to /home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/datasets/KITTI_RAW/trajectory_prediction/08/000973.bin
Processed and saved /home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/datasets/KITTI_RAW/trajectory_prediction/08/000974.json
Processed and saved 000974.bin to /home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/datasets/KITTI_RAW/trajectory_prediction/08/000974.bin
Processed and saved /home/bdi/huihuixu/after_ddos_0228/trajectory-prediction/datasets/KITTI_RAW/trajectory_prediction/08/000975.json
Processed and